<a href="https://colab.research.google.com/github/swarnapatro/AI_KickStart/blob/main/agentic_rag_ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Demo: Agentic RAG Pipeline (100% free - runs on Ollama in Google Colab)

**What will you build?**
A small "AI agent" that reads your question and **decides by itself** how to answer it:

| Question type | Tool the agent picks | Where the knowledge comes from |
|---|---|---|
| About our presentation (Agentic RAG, ReAct, Tool Use...) | `vector_search` | Our own text, stored in a FAISS vector database |
| Latest news / general facts | `web_search` | The internet (DuckDuckGo - free, no key needed) |
| Greetings / small talk | `general_conversation` | The LLM's own knowledge |

**Key words (plain English)**
- **LLM** - the "brain" that understands and writes text. Here: a free model running through **Ollama**.
- **Ollama** - a free program that runs open-source LLMs on the machine (here: the Colab machine). No API key, no billing.
- **RAG** (Retrieval-Augmented Generation) - first *find* useful text, then give it to the LLM so the answer is based on facts.
- **Vector store (FAISS)** - a searchable "library" where text is stored as numbers (embeddings) so we can search by *meaning*.
- **Agent / Router** - the part that decides *which tool to use*.
- **LangGraph** - a library that connects the steps (called *nodes*) into a flow (a *graph*).

**How the flow looks**
```
                 ┌──────────────┐
   Your question │    ROUTER    │  (LLM decides)
   ────────────► └──────┬───────┘
          ┌─────────────┼──────────────────┐
          ▼             ▼                  ▼
   vector_search    web_search     general_conversation
          └──────┬──────┘                  │
                 ▼                         │
         generate_response                 │
                 └──────────► END ◄────────┘
```

**Before you start:** `Runtime ▶ Change runtime type ▶ T4 GPU` (makes the LLM much faster). Then run the cells **one by one, top to bottom**.

## Step 0A - Install the Python libraries

In [1]:
# =====================================================================
# CELL 1 : Install Python libraries
# ---------------------------------------------------------------------
# WHAT IT DOES : Downloads and installs the Python packages used in this demo.
#                (Colab starts "empty" each time, so this is needed every session.)
# INPUT        : None
# OUTPUT       : Nothing important is shown (-q = quiet). Takes ~1 minute.
#                Warnings in red/orange about "dependency conflicts" can be ignored.
#
# WHY EACH PACKAGE?
#   langgraph             -> builds the agent flow (nodes + arrows)
#   langchain-core        -> basic building blocks (prompts, messages)
#   langchain-community   -> gives us the FAISS vector-store connector
#   langchain-ollama      -> lets LangChain talk to the Ollama LLM (chat + embeddings)
#   langchain-text-splitters -> cuts long text into small chunks
#   faiss-cpu             -> Facebook's fast vector search engine (our "library")
#   ddgs                  -> free DuckDuckGo web search (no API key needed)
# =====================================================================

!pip install -q langgraph langchain-core langchain-community langchain-ollama langchain-text-splitters faiss-cpu ddgs

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## Step 0B - Install Ollama (the program that runs the free LLM)

In [2]:
# =====================================================================
# CELL 2 : Install the Ollama program
# ---------------------------------------------------------------------
# WHAT IT DOES : Installs Ollama on the Colab machine.
#                Think of Ollama as "an app that can run AI models locally".
# INPUT        : None
# OUTPUT       : Installer messages ending with something like
#                ">>> The Ollama API is now available at 127.0.0.1:11434"
#                Takes ~1 minute.
#
# NOTE: Lines starting with "!" are run as terminal (shell) commands, not Python.
#   apt-get install ... -> tools the Ollama installer needs (zstd, pciutils, lshw)
#   curl ... | sh       -> downloads the official Ollama installer and runs it
# =====================================================================

!apt-get install -y -qq zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## Step 0C - Start Ollama and download the free models

In [3]:
# =====================================================================
# CELL 3 : Start the Ollama server and download two models
# ---------------------------------------------------------------------
# WHAT IT DOES : 1) Starts Ollama in the background (like opening an app).
#                2) Waits until it is ready.
#                3) Downloads two free models:
#                     - llama3.1:8b      = the chat LLM (the "brain", ~4.9 GB)
#                     - nomic-embed-text = the embedding model that turns text
#                                          into numbers for searching (~270 MB)
# INPUT        : Model names (set just below)
# OUTPUT       : "Ollama server is ready!" followed by download progress bars,
#                and finally the word "success" for each model.
#                First run takes ~3-5 minutes (the download is big). Be patient!
# =====================================================================

import subprocess   # lets Python start other programs (here: Ollama)
import time         # lets us pause / wait
import urllib.request  # lets us check whether the server answers

# --- Settings you can change later -----------------------------------
LLM_MODEL = "llama3.1:8b"          # chat model. Smaller/faster option: "llama3.2:3b"
EMBED_MODEL = "nomic-embed-text"   # embedding model used for vector search
# ----------------------------------------------------------------------

# Start "ollama serve" in the background so the notebook can keep running.
ollama_process = subprocess.Popen(
    ["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# Ask the server "are you ready?" every second (max 60 seconds).
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434")   # 11434 = Ollama's default port
        print("Ollama server is ready!")
        break
    except Exception:
        time.sleep(1)   # not ready yet -> wait 1 second and try again
else:
    raise RuntimeError("Ollama did not start. Re-run Cell 2 and Cell 3.")

# Download (pull) the models. Already-downloaded models are skipped automatically.
!ollama pull {LLM_MODEL}
!ollama pull {EMBED_MODEL}

Ollama server is ready!




## Step 1 - Import the tools we need

In [4]:
# =====================================================================
# CELL 4 : Imports
# ---------------------------------------------------------------------
# WHAT IT DOES : Loads (imports) the ready-made building blocks we installed.
#                An "import" is like taking a tool out of a toolbox.
# INPUT        : None
# OUTPUT       : Nothing printed. If there is no error, all is good.
# =====================================================================

# Python built-ins ------------------------------------------------------
from typing import TypedDict, Literal
# TypedDict -> lets us describe the "shape" of the data passed between steps
# Literal   -> restricts a value to a fixed list of allowed words
#              (e.g. the router may only answer "vector_search", "web_search"...)

# LLM + embeddings (Ollama) -----------------------------------------------
from langchain_ollama import ChatOllama, OllamaEmbeddings
# ChatOllama       -> the chat LLM (the "brain") that answers and decides
# OllamaEmbeddings -> converts text into numbers (vectors) so we can search by meaning

# Vector database ---------------------------------------------------------
from langchain_community.vectorstores import FAISS
# FAISS -> stores the vectors in memory and finds the most similar ones fast

# Text cutting --------------------------------------------------------------
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Cuts long text into small overlapping pieces ("chunks") - smaller pieces
# make search results more precise

# Prompt + data validation --------------------------------------------------
from langchain_core.prompts import ChatPromptTemplate
# ChatPromptTemplate -> a reusable instruction text with blanks like {query}
from pydantic import BaseModel, Field
# BaseModel, Field -> define the exact format we expect from the router's answer

# Web search ------------------------------------------------------------------
from ddgs import DDGS
# DDGS -> free DuckDuckGo search (no account / API key required)

# Agent flow -----------------------------------------------------------------
from langgraph.graph import StateGraph, END
# StateGraph -> the flowchart builder (add steps, connect with arrows)
# END        -> a special marker meaning "the flow finishes here"

print("All imports successful ✅")   # A yellow "DeprecationWarning" above this line is harmless - ignore it.

/tmp/ipykernel_1347/1988562255.py:22: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


All imports successful ✅


## Step 2 - Define the "State" (the agent's notebook)

The **state** is a small notebook that travels through every step of the flow.
Each step can read from it and write new notes into it.

In [5]:
# =====================================================================
# CELL 5 : Define the Graph State
# ---------------------------------------------------------------------
# WHAT IT DOES : Describes what information the agent carries from step to step.
# INPUT        : None (this only defines a structure)
# OUTPUT       : Nothing printed.
#
# The 4 "fields" of the notebook:
#   query     -> the user's question                       (set at the start)
#   next_node -> which step to run next (router's choice)  (set by router)
#   context   -> text found by vector/web search           (set by search steps)
#   response  -> the final answer                          (set by answer steps)
#
# total=False means "a field may be missing until some step fills it in".
# =====================================================================

class GraphState(TypedDict, total=False):
    query: str
    next_node: str
    context: str
    response: str

## Step 3 - Create the LLM (brain) and the web-search tool

In [ ]:
# =====================================================================
# CELL 6 : Create the LLM and the web search function
# ---------------------------------------------------------------------
# WHAT IT DOES : (1) Connects to the free Ollama model we downloaded.
#                (2) Defines a small helper that searches the internet.
# INPUT        : LLM_MODEL name (from Cell 3)
# OUTPUT       : A quick self-test: the LLM replies to "Say hello in 5 words."
#                Example output:  LLM says: Hello, it's nice to meet you!
# =====================================================================

# (1) The LLM ------------------------------------------------------------
llm = ChatOllama(
    model=LLM_MODEL,     # which Ollama model to use
    temperature=0,       # 0 = focused & repeatable answers (higher = more creative/random)
)

# (2) Web search helper --------------------------------------------------
def web_search(query: str, max_results: int = 3) -> list:
    """
    INPUT : a question, e.g. "latest AI news"
    OUTPUT: a list of text snippets from web pages, e.g.
            ["Anthropic released ...", "Google announced ...", "..."]
    """
    try:
        results = DDGS().text(query, max_results=max_results)   # ask DuckDuckGo
        return [r["body"] for r in results]                     # keep only the text part
    except Exception as error:
        print("Web search problem:", error)
        return []   # return an empty list so the demo keeps running

# (3) Self-test -----------------------------------------------------------
print("LLM says:", llm.invoke("Say hello in 5 words.").content)

## Step 4 - Build our private knowledge base (vector store)

We pretend this text is the content of a PowerPoint. We split it into pieces, turn each piece into numbers
(embeddings) and save them in FAISS. Later, the agent can search it **by meaning**.

In [ ]:
# =====================================================================
# CELL 7 : Build the vector store (our searchable knowledge base)
# ---------------------------------------------------------------------
# WHAT IT DOES : Text -> small chunks -> embeddings (numbers) -> FAISS database
#                -> a "retriever" (a search object) we can ask questions to.
# INPUT        : ppt_content (the text of our "presentation")
# OUTPUT       : Messages "Setting up..." / "complete", then the chunks created.
#                Example:  Created 2 chunks.
#                          Chunk 1: Agentic RAG: Transforming Knowledge Retrieval ...
#                          Then a mini-test: top result for "What is a ReAct Agent?"
# =====================================================================

# Our knowledge: replace this text with anything you like (notes, policies, FAQs...)
ppt_content = """
Agentic RAG: Transforming Knowledge Retrieval
Introduction to LLMs & RAG: The Foundation
LLMs have revolutionized how we interact with information. RAG enables access to external knowledge sources.
ReAct Agent: Iterative reasoning and action cycles for complete task satisfaction.
Tool Use Agent: Retrieves real-time data from APIs, SQL, external services.
"""

def setup_retriever(text: str):
    """
    INPUT : one long text (string)
    OUTPUT: a retriever - an object you can ask: retriever.invoke("your question")
            and it returns the 3 most relevant chunks.
    """
    print("Setting up the vector store retriever...")

    # Step A: cut the text into chunks.
    #   chunk_size=200    -> each piece is at most ~200 characters
    #   chunk_overlap=20  -> neighbouring pieces share 20 characters so no idea is cut in half
    splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
    chunks = splitter.split_text(text)                  # list of small strings
    print(f"Created {len(chunks)} chunks.")
    for i, chunk in enumerate(chunks, start=1):
        print(f"  Chunk {i}: {chunk[:70]!r}...")

    # Step B: choose the embedding model (text -> list of numbers)
    embeddings = OllamaEmbeddings(model=EMBED_MODEL)

    # Step C: embed every chunk and store the numbers in FAISS
    vector_store = FAISS.from_texts(texts=chunks, embedding=embeddings)

    # Step D: make a retriever. k=3 -> return the 3 closest chunks for each question
    print("Retriever setup complete.")
    return vector_store.as_retriever(search_kwargs={"k": 3})

# Build it once; later steps reuse the variable "retriever"
retriever = setup_retriever(ppt_content)

# Mini-test: search by meaning (no keyword match needed)
test_hits = retriever.invoke("What is a ReAct Agent?")
print("\nTop result for 'What is a ReAct Agent?':\n ", test_hits[0].page_content)

## Step 5 - The Agent's steps (nodes)

Each **node** is a normal Python function: it receives the *state* (the notebook) and returns what it wants to *add* to it.

### 5A - The Router: the part that "thinks" and picks a tool

In [ ]:
# =====================================================================
# CELL 8 : Router node  (the decision-maker of the agent)
# ---------------------------------------------------------------------
# WHAT IT DOES : Asks the LLM: "Which tool is best for this question?"
#                The LLM must answer with exactly one of 3 allowed words.
# INPUT        : state["query"]      e.g. "What is a ReAct Agent?"
# OUTPUT       : {"next_node": "vector_search"}   (or web_search / general_conversation)
#                It also prints:  Routing decision: vector_search
# =====================================================================

# The exact format we force the LLM to answer in: ONE of three words.
class RouteQuery(BaseModel):
    next_node: Literal["vector_search", "web_search", "general_conversation"] = Field(
        ..., description="Pick the best tool for the query."
    )

def router_node(state: GraphState):
    print("---NODE: ROUTER---")

    # Tell the LLM to reply in the RouteQuery format (no free-form text)
    structured_llm_router = llm.with_structured_output(RouteQuery)

    # The instructions (prompt). {state['query']} is replaced by the real question.
    system_prompt = f"""You are an expert routing agent. Route a user's query to the best tool.

Tools:
- vector_search: Use for presentation-specific info (Agentic RAG, RAG systems, LLMs, ReAct Agent, Tool Use Agent)
- web_search: Use for general, up-to-date info outside the presentation (news, recent events, facts)
- general_conversation: Use for greetings or casual chat

Query: {state['query']}
"""
    try:
        route = structured_llm_router.invoke(system_prompt)   # LLM thinks and decides
        decision = route.next_node
    except Exception as error:
        # Safety net: small models sometimes give a badly-formed answer.
        print("Router could not decide, using web_search. Reason:", error)
        decision = "web_search"

    print(f"Routing decision: {decision}")
    return {"next_node": decision}      # write the decision into the state

### 5B - The tool nodes: search our knowledge base, search the web, answer, chat

In [ ]:
# =====================================================================
# CELL 9 : The four "worker" nodes
# ---------------------------------------------------------------------
# WHAT IT DOES : Defines the steps that do the real work.
#
#   vector_search_node        IN : state["query"]
#                             OUT: {"context": "<3 best chunks from our PPT text>"}
#   web_search_node           IN : state["query"]
#                             OUT: {"context": "<snippets from web pages>"}
#   generate_response_node    IN : state["query"] + state["context"]
#                             OUT: {"response": "<final answer written by the LLM>"}
#   general_conversation_node IN : state["query"]
#                             OUT: {"response": "<friendly reply>"}
#
# Nothing runs yet - we are only defining the functions.
# =====================================================================

def vector_search_node(state: GraphState):
    print("---NODE: VECTOR SEARCH---")
    # Search our FAISS knowledge base by meaning -> returns the top 3 chunks
    retrieved_docs = retriever.invoke(state["query"])
    # Join the chunks into ONE text, separated by a line of dashes
    context = "\n---\n".join([doc.page_content for doc in retrieved_docs])
    print("Context found:", context[:150].replace("\n", " "), "...")
    return {"context": context}


def web_search_node(state: GraphState):
    print("---NODE: WEB SEARCH---")
    snippets = web_search(state["query"])          # helper from Cell 6
    context = "\n---\n".join(snippets)            # join snippets into ONE text
    print("Context found:", context[:150].replace("\n", " "), "...")
    return {"context": context}


def generate_response_node(state: GraphState):
    print("---NODE: GENERATE RESPONSE---")
    # Prompt template with 2 blanks: {query} and {context}.
    # "based only on the context" is what makes this RAG - the LLM must use the found facts.
    prompt = ChatPromptTemplate.from_template(
        """You are a helpful AI assistant. Answer the user's query based *only* on the context.
If context is empty or irrelevant, state you could not find the info.

User Query: {query}
Context:
---
{context}
---
Answer:"""
    )
    chain = prompt | llm      # "|" = pipe: fill the prompt, then send it to the LLM
    response = chain.invoke({"query": state["query"], "context": state.get("context", "")})
    return {"response": response.content}


def general_conversation_node(state: GraphState):
    print("---NODE: GENERAL CONVERSATION---")
    response = llm.invoke(state["query"])       # no search - the LLM just chats
    return {"response": response.content}

## Step 6 - Connect the steps into a flow (the graph)

In [ ]:
# =====================================================================
# CELL 10 : Build and compile the agent graph
# ---------------------------------------------------------------------
# WHAT IT DOES : Registers every node, then draws the arrows between them
#                (exactly like the diagram at the top of this notebook).
# INPUT        : The node functions from Cells 8 and 9
# OUTPUT       : A ready-to-run agent stored in the variable "app".
#                Prints: Agent graph is ready ✅
# =====================================================================

workflow = StateGraph(GraphState)        # an empty flowchart that uses our State

# 1) Add the boxes (nodes): "name shown in flow" -> function to run
workflow.add_node("router", router_node)
workflow.add_node("vector_search", vector_search_node)
workflow.add_node("web_search", web_search_node)
workflow.add_node("generate_response", generate_response_node)
workflow.add_node("general_conversation", general_conversation_node)

# 2) Where does the flow start? -> at the router
workflow.set_entry_point("router")

# 3) Conditional arrows: after "router", look at state["next_node"] and jump to that box
workflow.add_conditional_edges(
    "router",
    lambda state: state["next_node"],          # reads the router's decision
    {
        "vector_search": "vector_search",
        "web_search": "web_search",
        "general_conversation": "general_conversation",
    },
)

# 4) Normal arrows (always follow them)
workflow.add_edge("vector_search", "generate_response")   # found docs -> write answer
workflow.add_edge("web_search", "generate_response")      # found web text -> write answer
workflow.add_edge("generate_response", END)               # answer written -> finish
workflow.add_edge("general_conversation", END)            # chat reply -> finish

# 5) Lock the design and make it runnable
app = workflow.compile()
print("Agent graph is ready ✅")

## Step 7 - Run it! Ask the agent questions

Watch the printed lines: they show **which tool** the agent chose and **which steps** ran.

In [ ]:
# =====================================================================
# CELL 11 : Run demo queries
# ---------------------------------------------------------------------
# WHAT IT DOES : Sends 4 sample questions to the agent, one at a time.
#                Expected routing:
#                   Q1 (ReAct Agent)        -> vector_search       (our PPT text)
#                   Q2 (AI news Oct 2025)   -> web_search          (internet)
#                   Q3 (Hello...)           -> general_conversation (just chat)
#                   Q4 (Tool Use Agent)     -> vector_search       (our PPT text)
# INPUT        : A question (text)
# OUTPUT       : For every question: the steps that ran + the final answer.
#                Example:
#                   Executing query: 'Hello there, how are you doing?'
#                   ---NODE: ROUTER---
#                   Routing decision: general_conversation
#                   Finished node 'router'
#                   ...
#                   ✅ Final Answer: Hello! I'm doing well, thanks for asking...
#                Takes ~1-2 minutes in total. (Web results change daily, so Q2's answer will differ.)
# =====================================================================

def ask(question: str):
    """Send ONE question through the agent and print the steps + final answer."""
    print(f"\n{'=' * 50}\nExecuting query: '{question}'\n{'=' * 50}")

    final_state = {}
    # app.stream(...) runs the graph step by step and gives us each step's result
    for step in app.stream({"query": question}):
        for node_name, update in step.items():
            print(f"Finished node '{node_name}'")
            final_state.update(update)          # collect everything the steps wrote

    print("\n✅ Final Answer:")
    print(final_state["response"])


queries = [
    "What is a ReAct Agent according to the presentation?",
    "What are the latest advancements in AI in October 2025?",
    "Hello there, how are you doing?",
    "Tell me about the Tool Use Agent.",
]

for q in queries:
    ask(q)

## Step 8 - Your turn! 🎯

In [ ]:
# =====================================================================
# CELL 12 : Try your own question
# ---------------------------------------------------------------------
# WHAT IT DOES : Lets you type any question and see which tool the agent picks.
# INPUT        : Edit the text between the quotes below, then run the cell.
# OUTPUT       : The routing decision, the steps that ran and the final answer.
#
# Ideas to try:
#   "What does RAG enable?"                  -> should use vector_search
#   "Who won the last cricket world cup?"    -> should use web_search
#   "Tell me a joke"                         -> should use general_conversation
# =====================================================================

my_question = "What does RAG enable according to the presentation?"   # <-- change me

ask(my_question)

## 🎓 Recap

1. **Router** (LLM) read the question and chose a tool - that is what makes it *agentic*.
2. **Retrieval** fetched facts from our FAISS database or the web.
3. **Generation** - the LLM wrote the answer using only those facts.
4. Everything ran **free**: Ollama LLM + local embeddings + DuckDuckGo.

**Ideas to extend:** add your own text in Cell 7, try `LLM_MODEL = "llama3.2:3b"` in Cell 3 (faster, slightly less accurate), or add a new tool node (e.g. a calculator).